In [1]:
#1. Importar las librerías necesarias
import pandas as pd
import numpy as np
from sklearn import decomposition
from sklearn import preprocessing


In [2]:
#2. Leer el archivo CSV y cargarlo en un DataFrame

EmpleadosAttrition = pd.read_csv('empleadosReto.csv')
EmpleadosAttrition.head()

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,...,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,1,997,4,Male,...,22,4,3,80,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,1,178,2,Male,...,20,4,4,80,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,1,1780,2,Male,...,13,3,2,80,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,1,1118,2,Male,...,19,3,4,80,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,1,582,2,Male,...,12,3,4,80,15,2,4,6,7,Yes


In [3]:
#3a. Elimina la columna EmployeeCount del Datarame

EmpleadosAttrition = EmpleadosAttrition.drop(columns=['EmployeeCount'])


In [4]:
#3b. Elimina la columna EmployeeNumber del DataFrame

EmpleadosAttrition = EmpleadosAttrition.drop(columns=['EmployeeNumber'])


In [5]:
#3c. Elimina la columna Over18 del DataFrame

EmpleadosAttrition = EmpleadosAttrition.drop(columns=['Over18'])


In [6]:
#3d. Elimina la columna StandardHours del DataFrame

EmpleadosAttrition = EmpleadosAttrition.drop(columns=['StandardHours'])


In [7]:
#4a. Crea una columna llamada Year y obtén el año de contratación del empleado a partir de su fecha 'HiringDate'. Debe ser un entero.

EmpleadosAttrition['HiringDate'] = pd.to_datetime(EmpleadosAttrition['HiringDate'], format = 'mixed', errors = 'coerce')
EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].dt.year.astype('Int64')


In [8]:
#4b. Crea una columna llamada YearsAtCompany que contenga los años que el empleado lleva en la compañia hasta el año 2018. Usa la variable Year que se creó.

EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']


In [9]:
#5a. Renombra la variable DistanceFromHome a DistanceFromHome_km

EmpleadosAttrition = EmpleadosAttrition.rename(columns = {'DistanceFromHome': 'DistanceFromHome_km'})


In [10]:
#5b. Crea una nueva variable llamada DistanceFromHome que sea entera, es decir, solo con numeros.

EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.replace(' km', '').astype(int)


In [11]:
#6. Borra las columnas Year, HiringDate y DistanceFromHome_km

EmpleadosAttrition = EmpleadosAttrition.drop(columns = ['Year', 'HiringDate', 'DistanceFromHome_km'])


In [12]:
#7. Genera un nuevo frame llamado SueldoPromedioDpto que contenga MonthlyIncome promedio por departamendo de los empleados y colocalo en una variable llamado SueldoPromedio. Esta tabla solo es informativa, no la vas a utilizar en el set de datos que ests construyendo.

SueldoPromedioDpto = EmpleadosAttrition.groupby('Department')['MonthlyIncome'].mean()
SueldoPromedio = SueldoPromedioDpto.to_frame(name = 'SueldoPromedio')


In [13]:
#8. Escala la vairiable MonthlyIncome para que tenga un valor entre 0 y 1

escalador = preprocessing.MinMaxScaler()

EmpleadosAttrition['MonthlyIncome'] = escalador.fit_transform(EmpleadosAttrition[['MonthlyIncome']]) 


In [14]:
#9. Convierte todas las variables categóricas que quedan a númericas

for col in EmpleadosAttrition.columns:
    if EmpleadosAttrition[col].dtype == 'object':
        if 'Yes' in EmpleadosAttrition[col].values or 'No' in EmpleadosAttrition[col].values:
            EmpleadosAttrition[col] = EmpleadosAttrition[col].map({'Yes': 1, 'Y': 1, 'No': 0, 'N': 0})
        elif 'Male' in EmpleadosAttrition[col].values:
            EmpleadosAttrition[col] = EmpleadosAttrition[col].map({'Male': 1, 'Female': 0})

for col in ['BusinessTravel', 'MaritalStatus']:
    moda = EmpleadosAttrition[col].mode()[0]
    EmpleadosAttrition[col] = EmpleadosAttrition[col].fillna(moda)

EmpleadosAttrition = pd.get_dummies(EmpleadosAttrition, drop_first = True)

for col in EmpleadosAttrition.columns:
    EmpleadosAttrition[col] = pd.to_numeric(EmpleadosAttrition[col], errors = 'coerce')

EmpleadosAttrition = EmpleadosAttrition.fillna(0).astype(int)


In [15]:
#10. Calcula la correlación lineal entre cada una de las variables con respecto al Attrition

correlacion = EmpleadosAttrition.corr()

corr_attrition = correlacion['Attrition_Yes'].sort_values(ascending = False)


In [16]:
#11. Selecciona solo aquellas variables que tengan una correlación mayor o igual a 0.1, dejandolas en otro frame llamado EmpleadosAttritionFinal, manteniendo la variable de Salida Attrition_Yes

limite = 0.1

col_seleccionadas = corr_attrition[abs(corr_attrition) >= limite].index

EmpleadosAttritionFinal = EmpleadosAttrition[col_seleccionadas].copy()


In [17]:
#12. Crea una nueva variable llamada EmpleadosAttritionPCA formada por los componentes principales del frame EmpleadosAttritionFinal.

pca = decomposition.PCA()
scaler = preprocessing.StandardScaler()

y = EmpleadosAttritionFinal['Attrition_Yes']
X = EmpleadosAttritionFinal.drop('Attrition_Yes', axis = 1)

X_scaled = scaler.fit_transform(X)

EmpleadosAttritionPCA = pca.fit_transform(X_scaled)

print(f"Número de componentes seleccionados: {pca.n_components_}")

Número de componentes seleccionados: 14


In [18]:
#13. Agrega el mínimo numero de Componentes Principales en las columnas del frame EmpleadosAttritionPCA que logren explicar el 80% de la varianza al frame EmpleadosAttritionFinal

pca_80 = decomposition.PCA(n_components = 0.80)

pca_final = pca_80.fit_transform(X_scaled)

componentes_seleccionados = pca_80.n_components_

print(f"Número de componentes seleccionados para explicar el 80% de la varianza: {componentes_seleccionados}")

for i in range(componentes_seleccionados):
    nombre_col = f'C{i}'
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(**{nombre_col: pca_final[:, i]})


Número de componentes seleccionados para explicar el 80% de la varianza: 9


In [19]:
#14. Guarda el set de datos de EmpleadosAttritionFinal en un archivo csv llamado EmpleadosAttritionFinal.csv.

col_final = EmpleadosAttritionFinal.pop('Attrition_Yes')
EmpleadosAttritionFinal.insert(len(EmpleadosAttritionFinal.columns), 'Attrition_Yes', col_final)

EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index = False)

In [21]:
# Instala la herramienta de conversión
!pip install nbconvert
# Intenta generar el HTML de nuevo
!jupyter nbconvert --to html RetoEmpleados.ipynb

  Using cached nbconvert-7.17.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached bleach-6.3.0-py3-none-any.whl.metadata (31 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyterlab_pygments-0.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached mistune-3.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached nbclient-0.10.4-py3-none-any.whl.metadata (8.3 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached pandocfilters-1.5.1-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached webencodings-0.5.1-py2.py3-none-any.whl.metadata (2.1 kB)
  Using cached tinycss2-1.4.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-

In [22]:
import os
# Usamos 'python -m' para forzar a que use el nbconvert que acabamos de instalar
os.system('python -m jupyter nbconvert --to html RetoEmpleados.ipynb')
print("Proceso finalizado. ¡Busca el archivo HTML ahora!")

[NbConvertApp] Converting notebook RetoEmpleados.ipynb to html


Proceso finalizado. ¡Busca el archivo HTML ahora!


[NbConvertApp] Writing 309867 bytes to RetoEmpleados.html
